# Exploratory data analysis of the OTTO sample

Follows `_doc/eda_guide.md` step by step. One section per step; the decision log is `eda/eda_log.md`, comparison tables are in `eda/tables/`, figures in `eda/figures/`. Each step has its own GitHub issue (umbrella #5; step N is issue #91 + N). Rerun everything with `EDA_FORCE=1` (otherwise expensive tables are cached in `eda/tables/`).

**Two tables are analysed.**
- **Events** (`data/sample/events.parquet`): one row per event (`session`, `aid`, `ts`, `type`). No target. Used by the steps that do not need one.
- **Candidates** (`data/features/train_asof11.parquet`): one row per (session, candidate item) from the shared pool, with 18 numeric features, the category and the label. This is the modelling table for the steps that need a target.

In [ ]:
import platform
import sys
from pathlib import Path

import numpy as np
import polars as pl

ROOT = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "notebooks"))
from eda_utils import SEED, log_step, save_table
from split import DAY_MS, START_MS, day_to_ms

np.random.seed(SEED)
DATA = ROOT / "data"
D14, D16 = day_to_ms(14), day_to_ms(16)
events = pl.scan_parquet(DATA / "sample" / "events.parquet")
dev = events.filter(pl.col("ts") < D14)          # development set: every target-aware cell reads only this
CAND = DATA / "features" / "train_asof11.parquet"
pl.Config.set_tbl_rows(40); pl.Config.set_tbl_width_chars(200)

## Step 0. Setup, framing and holdout
*Kind: diagnostic. Issue #91.*

In [ ]:
framing = pl.DataFrame([
    ("Decision supported", "Rank candidate items for a session prefix and a query category (search and ranking on e-commerce sessions).", "given (project goal)"),
    ("Unit of analysis", "Events table: one event. Candidates table: one (session, candidate item) pair. The entity is the session; the same session has many rows in both.", "given"),
    ("Target", "`label > 0` on the candidates table: the candidate is clicked, carted or ordered later in the same session (graded: 1 click, 2 cart, 3 order). Defined in issues #10 and #22.", "assumed for the guide (the guide needs a single target column)"),
    ("Task type", "Binary classification on candidate rows (PR-AUC), and ranking (NDCG@10) for the final metric.", "assumed"),
    ("When is the target known", "After the session prefix: the held-out events that follow the cut.", "given"),
    ("Metric and error cost", "NDCG@10 with bootstrap intervals for ranking; PR-AUC for the classification view. Cost of false positives and false negatives not given.", "assumed"),
    ("Time order / future scoring", "Yes. Timestamps exist and the model scores later sessions.", "given"),
    ("Known at prediction time", "Features are computed as of a cutoff before the session (issue #22); the label is not.", "given"),
    ("Constraints", "8 GB RAM Mac, minimise cost, no PII (anonymous ids).", "given"),
], schema=["question", "answer", "source"], orient="row")
print(framing)

# Holdout: the evaluation window (sessions that start on days 14-15), time ordered because the model scores future data.
first = events.group_by("session").agg(pl.col("ts").min().alias("first_ts")).collect()
holdout = first.filter((pl.col("first_ts") >= D14) & (pl.col("first_ts") < D16))
dev_sessions = first.filter(pl.col("first_ts") < D14)
dev_max_ts = dev.select(pl.col("ts").max()).collect()[0, 0]
assert dev_max_ts < D14, "the development set must contain only events before day 14"
assert not set(holdout["session"].to_list()) & set(dev_sessions["session"].to_list())
cand_sessions = pl.read_parquet(CAND, columns=["session"])["session"].unique()
cand_first = first.filter(pl.col("session").is_in(cand_sessions.to_list()))
assert (cand_first["first_ts"] < D14).all() and (cand_first["first_ts"] >= day_to_ms(11)).all()
import lightgbm
import scipy
import sklearn

versions = {"python": platform.python_version(), "polars": pl.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
            "scikit-learn": sklearn.__version__, "lightgbm": lightgbm.__version__, "seed": SEED}
print(f"holdout: {holdout.height:,} sessions starting on days 14-15 (time-ordered); development: {dev_sessions.height:,} sessions, events before day 14")
print("candidates table sessions (all development):", cand_sessions.len())
print(versions)

In [ ]:
e0 = log_step(0, "Setup and framing", "Diagnostic",
    "Framing table above (each answer marked given or assumed). Holdout scheme options: time-ordered, grouped by entity, random. Layout eda/{tables,figures}, seed 42, versions printed.",
    f"Development events: ts < day 14 ({dev_sessions.height:,} sessions). Holdout: {holdout.height:,} sessions that start on days 14-15. No session is on both sides. The candidates table has {cand_sessions.len():,} sessions, all in development (they start on days 11-13).",
    "Time-ordered holdout = the evaluation window of issue #6 (days 14-15). Everything target-aware reads only events before day 14.",
    "The model scores future sessions, so the latest rows are held out (guide rule). Grouping by session is implied: holdout sessions start after the development window.",
    "Random split: rejected, it would mix time periods. Group-only split: rejected, it does not test future data.",
    "High", "Steps 12 and 15 re-check the scheme; the holdout is never read before the modelling phase.",
    "yes: the target and task type are assumed from issues #10/#22. Please confirm `label > 0` is the target to use for the guide.")
print(e0)

## Step 1. Load and first look
*Kind: diagnostic. Issue #92.* Options: A manual inspection, B per-column overview, C automated profile.

In [ ]:
ev = dev.collect()                                          # 13.06 million rows x 4 columns: fits in memory
print("A. shape:", ev.shape, " estimated memory:", round(ev.estimated_size() / 1e6), "MB")
print("dtypes:", dict(ev.schema))
print("first 5:\n", ev.head(5)); print("last 5:\n", ev.tail(5)); print("random 5:\n", ev.sample(5, seed=SEED))
print(ev.describe())

In [ ]:
overview = pl.DataFrame([{
    "column": c, "dtype": str(ev.schema[c]), "missing_share": ev[c].null_count() / ev.height, "n_unique": ev[c].n_unique(),
    "unique_share": ev[c].n_unique() / ev.height, "example": str(ev[c][0]), "min": str(ev[c].min()), "max": str(ev[c].max()),
} for c in ev.columns])
save_table("step01_column_overview", overview)
print("B. per-column overview:"); print(overview)
print("C. automated profile: not run, see the log (ydata-profiling needs numpy<2.2 and matplotlib<=3.10; the project has numpy",
      np.__version__, ")")

In [ ]:
issues1 = pl.DataFrame([
    ("session", "ID-like: 1.0 million distinct ids, an identifier not a feature", "exclude from features; use as the entity for splits (Steps 4, 15)"),
    ("aid", "ID-like item identifier with 1.4 million distinct values, long tail", "never a numeric feature; encoding options in Step 13"),
    ("ts", "Unix milliseconds stored as int64", "convert to a datetime for time steps (Step 2A)"),
    ("type", "int8 code 0/1/2 for click/cart/order, low cardinality", "treat as categorical (Step 2A)"),
    ("all", "No missing values, no constant column", "Step 5 has little to treat"),
], schema=["column", "problem", "handled_in"], orient="row")
save_table("step01_issue_list", issues1); print(issues1)
assert ev.null_count().sum_horizontal()[0] == 0
e1 = log_step(1, "Load and first look", "Diagnostic",
    "A manual inspection (shape, dtypes, first/last/random rows, describe); B per-column overview table; C automated profile SKIPPED: ydata-profiling requires numpy<2.2 and matplotlib<=3.10, the project uses numpy 2.4.6 and matplotlib 3.11.2, so installing it would downgrade core libraries for every earlier result; with 4 ID-like columns A and B already show its alerts.",
    f"{ev.height:,} rows x 4 columns (session int64, aid int32, ts int64, type int8), {ev.estimated_size() / 1e6:.0f} MB; no missing values; no constant column; session and aid are ID-like; ts is epoch milliseconds; type has 3 codes.",
    "Prioritised issue list saved to eda/tables/step01_issue_list.csv: convert ts, treat type as categorical, exclude ids from features.",
    "The three options were meant to be combined into one list; two were run and their findings agree.",
    "C profile: skipped for the reason above; installing at the cost of downgrading numpy was rejected.",
    "High", "Step 2 converts types; Step 13 handles aid as high-cardinality.")
print(e1)

## Step 2. Types, validity and consistency
*Kind: diagnostic then fixes. Issue #93.* Options: A dtype vs meaning, B sentinel values, C range and domain rules, D cross-field consistency (development events, ts < day 14).

In [ ]:
START, END = START_MS, START_MS + 28 * DAY_MS
n = ev.height
# A dtype vs meaning
A = pl.DataFrame([
    ("session", "int64", "identifier", "keep as id, never a numeric feature"),
    ("aid", "int32", "identifier", "keep as id"),
    ("ts", "int64", "epoch milliseconds", "convert to datetime for time steps; the raw integer is kept for splits"),
    ("type", "int8", "categorical code 0 click, 1 cart, 2 order", "cast to a categorical for encoders; the code is fine for trees"),
], schema=["column", "stored_as", "meaning", "conversion"], orient="row"); print("A.\n", A)
# B sentinels
sent = [-1, 0, 999, 9999, 99999]
hits = [{"column": c, "value": v, "rows": int((ev[c].cast(pl.Int64) == v).sum())} for c in ("session", "aid", "ts", "type") for v in sent]
B = pl.DataFrame([h for h in hits if h["rows"] > 0] or [{"column": "-", "value": 0, "rows": 0}]).with_columns((pl.col("rows") / n).alias("share")); print("B. sentinel-like values found:\n", B)
# C ranges
C = pl.DataFrame([
    ("ts inside the 4 labelled weeks", int(((ev["ts"] < START) | (ev["ts"] >= END)).sum())),
    ("type not in {0,1,2}", int((~ev["type"].is_in([0, 1, 2])).sum())),
    ("aid < 0", int((ev["aid"] < 0).sum())), ("session < 0", int((ev["session"] < 0).sum())),
    ("ts in the future of the development cut (>= day 14)", int((ev["ts"] >= D14).sum())),
], schema=["rule", "violations"], orient="row").with_columns((pl.col("violations") / n).alias("share")); print("C.\n", C)
# D cross-field
srt = ev.select(pl.col("session"), pl.col("ts"), (pl.col("ts").diff() < 0).alias("backwards"), (pl.col("session") != pl.col("session").shift(1)).alias("new"))
backwards_within = int((srt["backwards"] & ~srt["new"]).sum())
blocks = int(srt["new"].sum()) + 1; sessions = ev["session"].n_unique()   # +1: the first row has no previous row
first_seen = ev.group_by("session", "aid").agg(
    pl.col("ts").filter(pl.col("type") == 0).min().alias("t_click"), pl.col("ts").filter(pl.col("type") == 1).min().alias("t_cart"),
    pl.col("ts").filter(pl.col("type") == 2).min().alias("t_order"))
cart_no_click = first_seen.filter(pl.col("t_cart").is_not_null() & (pl.col("t_click").is_null() | (pl.col("t_cart") < pl.col("t_click")))).height
order_no_cart = first_seen.filter(pl.col("t_order").is_not_null() & (pl.col("t_cart").is_null() | (pl.col("t_order") < pl.col("t_cart")))).height
pairs = first_seen.height
D = pl.DataFrame([
    ("timestamps go backwards inside a session", backwards_within, backwards_within / n),
    ("a session appears in more than one block (blocks - sessions)", blocks - sessions, (blocks - sessions) / n),
    ("a cart of an item with no earlier click (share of session-item pairs)", cart_no_click, cart_no_click / pairs),
    ("an order of an item with no earlier cart (share of session-item pairs)", order_no_cart, order_no_cart / pairs),
], schema=["rule", "violations", "share"], orient="row"); print("D.\n", D)
for name, t in (("A", A), ("B", B), ("C", C), ("D", D)):
    save_table(f"step02_{name}", t)

In [ ]:
bad_share = max(float(C["share"].max()), float(D["share"][:2].max()))
e2 = log_step(2, "Types, validity and consistency", "Diagnostic then fixes",
    "A dtype vs meaning; B sentinel-like values (-1, 0, 999, 9999, 99999); C range and domain rules; D cross-field consistency (ordering inside sessions, cart/order after click/cart). All four run on the development events.",
    f"A: 4 columns, all correctly typed except that ts is epoch ms. B: no placeholder found. The scan flags aid values 0, 999, 9999 and 99999 (2, 4, 44 and 1 events), but item ids are dense integers from 0 to 1,855,602, so these are ordinary ids; session 0 (86 events) is the first session; type 0 is the code for a click (89.8% of events). Details in eda/tables/step02_B.csv. C: {int(C['violations'].sum()):,} range violations (ts inside the 4 weeks, type in {{0,1,2}}, ids non-negative). D: {backwards_within:,} backwards timestamps inside sessions; {cart_no_click / pairs:.1%} of carted items had no earlier click in the same session and {order_no_cart / pairs:.1%} of ordered items had no earlier cart.",
    "No fixes needed for A-C. D is a property of the data (users can add to the cart from a page without a logged click), not an error. The sentinel-like values are not placeholders (evidence above), so nothing goes on the human question list.",
    "Rule: apply unambiguous fixes, convert impossible values to missing with a flag, never delete rows. There was nothing impossible.",
    "Nothing to reject: all impossible-value options found no violations. Treating cart-without-click as an error was rejected because it is common and plausible.",
    "High", "Step 4 uses the key (session, aid, ts, type); Step 12 must not assume a click precedes a cart.")
print(e2)

## Step 3. Target analysis
*Kind: diagnostic. Issue #94.* Target on the candidates table: `label > 0`.

In [ ]:
cand = pl.read_parquet(CAND, columns=["session", "aid", "category", "label", "y_click", "y_cart", "y_order"])
print("A. label counts and shares:")
A3 = cand.group_by("label").len().sort("label").with_columns((pl.col("len") / cand.height).alias("share")); print(A3)
pos = int((cand["label"] > 0).sum())
print(f"y = label > 0: {pos:,} positives of {cand.height:,} rows ({pos / cand.height:.3%}); imbalance ratio 1 : {(cand.height - pos) / pos:.0f}")
B3 = pl.DataFrame({"class": ["any (label>0)", "click", "cart", "order"], "positives": [pos, int(cand["y_click"].sum()), int(cand["y_cart"].sum()), int(cand["y_order"].sum())]}); print("B. absolute minority counts:"); print(B3)
first_ts = first.filter(pl.col("session").is_in(cand["session"].unique().to_list()))
c2 = cand.join(first_ts, on="session").with_columns(((pl.col("first_ts") - day_to_ms(0)) // DAY_MS).alias("day"))
C3a = c2.group_by("day").agg(pl.len().alias("rows"), (pl.col("label") > 0).mean().alias("target_rate")).sort("day"); print("C. by day of the session start:"); print(C3a)
top5 = c2.group_by("category").len().sort("len", descending=True).head(5)["category"].to_list()
C3b = c2.filter(pl.col("category").is_in(top5)).group_by("category").agg(pl.len().alias("rows"), (pl.col("label") > 0).mean().alias("target_rate")).sort("rows", descending=True); print("C. by the 5 largest categories:"); print(C3b)
print("D. rows without a target:", int(cand["label"].null_count()))
for nme, t in (("A", A3), ("B", B3), ("C_day", C3a), ("C_category", C3b)):
    save_table(f"step03_{nme}", t)

In [ ]:
# minority counts in the development set
dev_pos = pos  # the candidates table contains development sessions only
e3 = log_step(3, "Target analysis", "Diagnostic",
    "A class counts and shares; B absolute minority counts; C target rate by day and by the 5 largest categories; D missing target.",
    f"{cand.height:,} rows, {pos:,} positives ({pos / cand.height:.3%}); ratio 1:{(cand.height - pos) / pos:.0f}. Positives by type: click {int(cand['y_click'].sum()):,}, cart {int(cand['y_cart'].sum()):,}, order {int(cand['y_order'].sum()):,}. Target rate by start day and by category in eda/tables/step03_C_*.csv. No missing target.",
    "Class imbalance is SEVERE (minority under 1 percent): Step 14 handles it. The cart and order classes are small but above 100 rows; orders are the rarest.",
    "Guide rules: flag imbalance under 10 percent, severe under 1 percent; remove rows with no target (none).",
    "No option was rejected; all four were run.",
    "High", "Step 14 must use PR-AUC; Step 15 uses grouped folds because a session has 200 rows; the order task has few positives, so per-task results are noisy.")
print(e3)

## Step 4. Duplicates
*Kind: selection (how to treat them). Issue #95.* Detection: A exact, B key-based, C near duplicates, D conflicting labels.

In [ ]:
nrows = ev.height
# A exact
A4 = nrows - ev.unique().height
# B key based (session, aid, ts) and (session, aid, type, ts)
B1 = nrows - ev.select("session", "aid", "ts").unique().height
B2 = nrows - ev.select("session", "aid", "type", "ts").unique().height
# C near duplicates: same session, item and type within the same second
C4 = nrows - ev.with_columns((pl.col("ts") // 1000).alias("sec")).select("session", "aid", "type", "sec").unique().height
dup = pl.DataFrame([("A exact (all 4 columns)", A4), ("B key (session, aid, type, ts)", B2), ("B key (session, aid, ts), any type", B1), ("C near (session, aid, type, same second)", C4)],
    schema=["kind", "rows_beyond_the_first"], orient="row").with_columns((pl.col("rows_beyond_the_first") / nrows).alias("share")); print(dup)
save_table("step04_events_duplicates", dup)
# how do the exact duplicates cluster?
exact = ev.group_by(ev.columns).len().filter(pl.col("len") > 1)
print("groups with exact duplicates:", exact.height, "; largest group size:", int(exact["len"].max()) if exact.height else 0)
by_type = ev.filter(pl.struct(ev.columns).is_duplicated()).group_by("type").len().sort("type"); print("exact duplicates by event type (all copies):"); print(by_type)
# across the development / holdout boundary: sessions on both sides (none, checked in Step 0)
assert not set(holdout["session"].to_list()) & set(dev_sessions["session"].to_list())

In [ ]:
# D conflicting labels on the candidates table: identical (rounded) features but different target
feat = pl.read_parquet(CAND).drop(["session", "aid"])
num = [c for c in feat.columns if feat.schema[c] in (pl.Float32, pl.Float64)]
rounded = feat.with_columns([pl.col(c).round(3) for c in num])
keys = [c for c in rounded.columns if c not in ("label", "y_click", "y_cart", "y_order")]
groups = rounded.with_columns((pl.col("label") > 0).alias("y")).group_by(keys).agg(pl.len().alias("n"), pl.col("y").n_unique().alias("n_targets"), pl.col("y").sum().alias("pos"))
multi = groups.filter(pl.col("n") > 1)
conflict = multi.filter(pl.col("n_targets") > 1)
rows_in_conflict = int(conflict["n"].sum()); minority_in_conflict = int(conflict["pos"].sum())
print(f"D. feature combinations shared by 2+ rows: {multi.height:,} ({int(multi['n'].sum()):,} rows); with conflicting targets: {conflict.height:,} combinations, {rows_in_conflict:,} rows ({rows_in_conflict / feat.height:.2%} of all rows)")
print("share of positives that sit in a conflicting group:", round(minority_in_conflict / pos, 3))
D4 = pl.DataFrame({"measure": ["combinations shared by 2+ rows", "rows in those combinations", "combinations with conflicting targets", "rows in conflicting combinations", "share of all rows", "share of positives in a conflict"],
                   "value": [float(multi.height), float(multi["n"].sum()), float(conflict.height), float(rows_in_conflict), rows_in_conflict / feat.height, minority_in_conflict / pos]})
save_table("step04_candidates_conflicts", D4); print(D4)
del feat, rounded, groups

In [ ]:
order_dups = int(by_type.filter(pl.col("type") == 2)["len"][0]); click_dups = int(by_type.filter(pl.col("type") == 0)["len"][0]); cart_dups = int(by_type.filter(pl.col("type") == 1)["len"][0])
n_order = int((ev["type"] == 2).sum())
print(f"order events involved in exact duplicates: {order_dups:,} of {n_order:,} ({order_dups / n_order:.1%}); clicks {click_dups:,} of {int((ev['type'] == 0).sum()):,} ({click_dups / int((ev['type'] == 0).sum()):.2%}); carts {cart_dups:,}")
e4 = log_step(4, "Duplicates", "Selection (how to treat them)",
    "A exact; B key-based ((session, aid, type, ts) and (session, aid, ts)); C near-duplicates (same session, item, type in the same second); D conflicting labels on the candidates table (features rounded to 3 decimals). Treatments considered: keep everything, drop exact duplicates keeping the first, drop only the click and cart duplicates, aggregate per key.",
    f"A: {A4:,} rows ({A4 / nrows:.3%}) are exact copies of an earlier row, in {exact.height:,} groups (largest group 6). B is identical to A (no key repeats with a different value or type). C: {C4:,} rows ({C4 / nrows:.3%}). By event type, the copies are mostly ORDERS: {order_dups:,} order events ({order_dups / n_order:.1%} of all order events) are involved, against {click_dups:,} clicks and {cart_dups:,} carts. D: {conflict.height} feature combinations have conflicting targets, covering {rows_in_conflict} rows ({rows_in_conflict / cand.height:.5%} of {cand.height:,}); each holds one positive and one negative, i.e. {minority_in_conflict} positives ({minority_in_conflict / pos:.2%} of all positives). Duplicates across the development/holdout boundary: none (no session is on both sides).",
    "Drop exact duplicate CLICK and CART events, keeping the first (double logging in the same millisecond has no business reason). KEEP order duplicates: an order of the same item logged twice at the same millisecond is most likely a quantity above 1, which is real. Conflicting labels: none to treat. Not applied to the data used by the earlier issues (#7, #17-#29): the events involved are 0.10% of clicks and 0.09% of carts (copies including the first occurrence), and the effect on those results is measured in follow-up issue #109.",
    "Guide rules: exact duplicates with no business reason are dropped keeping the first; legitimate repeated observations are kept; conflicting labels are quantified and never deleted without approval. Orders have a plausible business reason (quantity), clicks and carts do not.",
    "Keep everything: rejected for clicks and carts (they would double count in co-visitation and popularity). Drop all exact duplicates: rejected because it would remove real order quantities (8.7% of order events are involved). Aggregate per key: rejected, it changes the event grain that Steps 12 and 16 rely on. Near-duplicates (C): NOT dropped: a second click on the same item within one second cannot be told from a real fast repeat without inspecting a sample, and the guide says drop near duplicates only if a sample shows they are the same record.",
    "Medium (the order-quantity reading is an inference, not something the data proves)",
    "Affects the event counts by 0.15%; co-visitation, popularity and features from earlier issues used the un-deduplicated events. Follow-up issue #109 quantifies the effect on the closed results. Step 15 uses grouped folds by session regardless.")
print(e4)